In [ ]:
                        # AI-powered customer support system for a fintech company

In [9]:
#  Question 2: Design a Multi-Agent Workflow with LangGraph (25 Marks)
# 🧩 Scenario
# You are building an AI-powered customer support system for a fintech company.
# The system must handle:
#     Transaction queries
#     Fraud detection flags
#     Refund requests
#     General FAQs
# The company wants:
#     High accuracy
#     Step-by-step reasoning
#     Ability to retry if answer is incorrect
#     Modular, scalable architecture
# 💻 Task
# Design and implement a multi-agent workflow using LangGraph (or similar framework).
# ✅ 1. Agent Design
# Define at least 3 agents, such as:
#     Retrieval Agent
#     Reasoning/Answer Agent
#     Validation Agent
# Explain briefly (in comments or code):
#     Each agent’s role
#     Input/output
# ✅ 2. Graph Workflow Implementation
# Write code or pseudo-code to:
#     Define state
#     Add nodes (agents)
#     Define edges
#     Implement conditional logic
# 👉 Must include:
#     Retry loop if validation fails
#     Clear start and end states
# ✅ 3. State Management
# Show how state evolves across steps:
#     Query
#     Context
#     Intermediate reasoning
#     Final answer
#     Validation flag
# ✅ 4. Task Delegation & Communication
# Demonstrate:
#     How agents pass information
#     How decisions are made between agents
# 🎯 Expected Outcome
# A clear multi-step, graph-based agent system that:

#     Handles complex queries
#     Demonstrates reasoning + validation
#     Uses proper orchestration 

import os
import requests
from dotenv import load_dotenv
from langgraph.graph import StateGraph

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("Please set GROQ_API_KEY in .env file")

MOCK_MODE = True

def groq_generate(prompt, model="groq/llama-3.3-70b-versatile"):
    if MOCK_MODE:
        # Offline testing with mock responses
        if "refund" in prompt.lower():
            return "Refunds are processed within 5-7 business days."
        elif "fraud" in prompt.lower():
            return "Fraud transactions are flagged automatically and reviewed by our team."
        elif "transaction" in prompt.lower():
            return "Transactions are recorded instantly."
        else:
            return "I don't know, please contact support."
    # Real API call
    url = "https://api.groq.ai/v1/completions"
    headers = {"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type": "application/json"}
    payload = {"model": model, "prompt": prompt, "max_tokens": 200}
    try:
        response = requests.post(url, json=payload, headers=headers, timeout=10)
        response.raise_for_status()
        return response.json().get("choices", [{}])[0].get("text", "").strip()
    except requests.exceptions.RequestException as e:
        return f"[Error: {e}]"


def retrieval_agent(state):
    """Fetch relevant context based on query keywords"""
    query = state.get("query", "")
    if "refund" in query.lower():
        context = "Refunds are processed in 5-7 business days."
    elif "fraud" in query.lower():
        context = "Fraudulent transactions are flagged automatically."
    elif "transaction" in query.lower():
        context = "All transactions are logged instantly."
    else:
        context = "General FAQ: Please contact support."
    state["context"] = context
    return state

def reasoning_agent(state):
    """Generate answer using Groq (or mock)"""
    query = state.get("query", "")
    context = state.get("context", "")
    prompt = f"Answer the customer query using only this context:\nContext: {context}\nQuestion: {query}"
    state["answer"] = groq_generate(prompt)
    return state

def validation_agent(state):
    """Validate answer; retry if generic/failure"""
    answer = state.get("answer", "")
    retry_count = state.get("retry_count", 0)
    # Consider valid if answer is not empty and not generic
    state["valid"] = not ("contact support" in answer.lower() or answer.strip() == "")
    state["retry_count"] = retry_count + 1
    return state

graph = StateGraph(dict)
graph.add_node("retrieval", retrieval_agent)
graph.add_node("reasoning", reasoning_agent)
graph.add_node("validation", validation_agent)

graph.set_entry_point("retrieval")
graph.add_edge("retrieval", "reasoning")
graph.add_edge("reasoning", "validation")

# Conditional edge for retry / end
def decide_next(state):
    return "retrieval" if not state["valid"] and state["retry_count"] < 2 else "end"

graph.add_conditional_edges("validation", decide_next, {"retrieval": "retrieval", "end": "__end__"})
app = graph.compile()

queries = [
    "How long does a refund take?",
    "What happens if I detect a fraud?",
    "How do I check transaction status?",
    "What is the customer support number?"
]

for q in queries:
    print(f"\n=== QUERY: {q} ===")
    result = app.invoke({"query": q, "retry_count": 0})
    for k, v in result.items():
        print(f"{k}: {v}")


=== QUERY: How long does a refund take? ===
query: How long does a refund take?
retry_count: 1
context: Refunds are processed in 5-7 business days.
answer: Refunds are processed within 5-7 business days.
valid: True

=== QUERY: What happens if I detect a fraud? ===
query: What happens if I detect a fraud?
retry_count: 1
context: Fraudulent transactions are flagged automatically.
answer: Fraud transactions are flagged automatically and reviewed by our team.
valid: True

=== QUERY: How do I check transaction status? ===
query: How do I check transaction status?
retry_count: 1
context: All transactions are logged instantly.
answer: Transactions are recorded instantly.
valid: True

=== QUERY: What is the customer support number? ===
query: What is the customer support number?
retry_count: 2
context: General FAQ: Please contact support.
answer: I don't know, please contact support.
valid: False
